<a href="https://colab.research.google.com/github/lvndrmlk/ml-task/blob/main/ml_red_task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate transformers trl datasets peft

In [2]:
import os
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen3-4B-Instruct-2507"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)

print("ура пабеда")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

ура пабеда


In [5]:
from datasets import load_dataset

dataset = load_dataset('json', data_files='kid_adult.jsonl', split='train')

def make_text(example):
    messages = [
        {'role': 'user', 'content': example['prompt']},
        {'role': 'assistant', 'content': example['kid']},
    ]
    return {'text': tokenizer.apply_chat_template(messages, tokenize=False)}

dataset = dataset.map(make_text)

print('train rows:', len(dataset))
print(dataset[0]['text'][:500])


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1489 [00:00<?, ? examples/s]

train rows: 1489
<|im_start|>user
Как работает камера видеонаблюдения? Ответ: Она ловит свет через стеклышко, превращает его в электрические сигналы и сохраняет их как видео в памяти.<|im_end|>
<|im_start|>assistant
Камера работает как волшебный глаз. Она улавливает свет через свое стеклышко, переводит его в электрические сигналы и сохраняет как видео в памяти.<|im_end|>



In [6]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=512)

tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir='./task1_sft',
    seed=42,
    data_seed=42,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy='no',
    report_to='none',
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer.train()


Map:   0%|          | 0/1489 [00:00<?, ? examples/s]

trainable params: 5,898,240 || all params: 4,028,366,336 || trainable%: 0.1464


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,2.307053
20,1.622692
30,1.484197
40,1.454936
50,1.457660
60,1.390636
70,1.358714
80,1.336335
90,1.350434
100,1.342081


TrainOutput(global_step=187, training_loss=1.4197308035457836, metrics={'train_runtime': 1419.0354, 'train_samples_per_second': 1.049, 'train_steps_per_second': 0.132, 'total_flos': 4427779208325120.0, 'train_loss': 1.4197308035457836, 'epoch': 1.0})

In [8]:
import json
import torch
from tqdm.notebook import tqdm
from transformers import set_seed

set_seed(42)
peft_model.eval()

test_dataset = []
with open('public_test_style.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        test_dataset.append(json.loads(line))

generated_responses = []

for item in tqdm(test_dataset):
    messages = [{'role': 'user', 'content': item['prompt']}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(peft_model.device)

    with torch.no_grad():
        output = peft_model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    answer = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    generated_responses.append(answer.strip())

print('тестовые строки:', len(test_dataset))
print('пример:')
print(generated_responses[0])


  0%|          | 0/50 [00:00<?, ?it/s]

тестовые строки: 50
пример:
Продавцы делают скидки, чтобы быстро продать старые вещи и освободить место для новых. Это как если бы ты забрал старую игрушку и купил новую — так они хотят быстро продать старые товары и купить новые.


In [13]:
import pickle
import numpy as np
from scipy.sparse import hstack

with open('style_clf.pkl', 'rb') as f:
    style_data = pickle.load(f)

clf = style_data['clf']
vecs = style_data['vecs']

x1 = vecs[0].transform(generated_responses)
x2 = vecs[1].transform(generated_responses)
x = hstack([x1, x2])

p_simple = clf.predict_proba(x)[:, 1]
mean_p_simple = float(np.mean(p_simple))


print('P_simple:', round(mean_p_simple, 4))


P_simple: 0.9769
